# Claude Evaluation - Extended Thinking with Token Logging


**Install:** `pip install anthropic`

In [ ]:
import anthropic

api_key = "YOUR_ANTHROPIC_API_KEY"  # ← Your Anthropic API key
client = anthropic.Anthropic(api_key=api_key)

In [ ]:
import pandas as pd

df = pd.read_csv("Benchmark/OP_Benchmark.csv")  
print(f"Loaded {len(df)} questions")
df.head()

In [ ]:
import time, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============ CONFIGURATION ============
# Model options:
# - "claude-opus-4-6" 
# - "claude-opus-4-5-20251101" 
# - "claude-sonnet-4-20250514" 

model_name = "claude-opus-4-6"  # Change to claude-opus-4-5-20251101 for manual thinking

# Extended thinking configuration
# For Opus 4.6/4.5: Use adaptive thinking (effort controls depth)
# For Sonnet 4.5: Use manual thinking (budget_tokens + effort)
USE_ADAPTIVE_THINKING = True  # True for 4.6, False for 4.5
THINKING_BUDGET = 16384  # Only used if USE_ADAPTIVE_THINKING=False 
MAX_TOKENS = 64000
EFFORT_LEVEL = "max"  # Options: "low", "medium", "high", "max" (max only for Opus 4.6)

CSV_PATH = f"{model_name.replace('-', '_')}_effort_{EFFORT_LEVEL}_OP_Benchmark.csv"

MAX_WORKERS = 4
RPM_LIMIT = 40
SAVE_EVERY = 5
MAX_RETRIES = 10  # Max retries for empty responses

# ============ COLUMNS FOR LOGGING ============
if 'answer_to_prompt_1' not in df.columns:
    df['answer_to_prompt_1'] = None
if 'answer_to_prompt_2' not in df.columns:
    df['answer_to_prompt_2'] = None
if 'input_tokens_1' not in df.columns:
    df['input_tokens_1'] = None
if 'input_tokens_2' not in df.columns:
    df['input_tokens_2'] = None
if 'output_tokens_1' not in df.columns:
    df['output_tokens_1'] = None
if 'output_tokens_2' not in df.columns:
    df['output_tokens_2'] = None
# Thinking tokens estimated from summary
if 'thinking_tokens_1' not in df.columns:
    df['thinking_tokens_1'] = None
if 'thinking_tokens_2' not in df.columns:
    df['thinking_tokens_2'] = None
# Save the actual thinking content (summary text)
if 'thinking_content_1' not in df.columns:
    df['thinking_content_1'] = None
if 'thinking_content_2' not in df.columns:
    df['thinking_content_2'] = None

df.reset_index(drop=True, inplace=True)

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)
save_lock = threading.Lock()

def save():
    with save_lock:
        df.to_csv(CSV_PATH, index=False)

# Track refused/failed questions
refused_questions = []

# ============ API CALL - RETURNS ANSWER + TOKEN USAGE + THINKING CONTENT ============
def ask_until_success(prompt, row_idx=None):
    """Returns: (answer_text, input_tokens, output_tokens, thinking_tokens_est, thinking_content)"""
    retry_count = 0
    
    while retry_count < MAX_RETRIES:
        try:
            rpm_limiter.wait()
            
            # Configure thinking based on model version
            if USE_ADAPTIVE_THINKING:
                # Opus 4.6: Adaptive thinking (effort controls depth)
                thinking_config = {"type": "adaptive"}
            else:
                # Opus 4.5: Manual thinking with budget
                thinking_config = {
                    "type": "enabled",
                    "budget_tokens": THINKING_BUDGET
                }
            
            # Use streaming for extended thinking (required for long requests)
            request_kwargs = {
                "model": model_name,
                "max_tokens": MAX_TOKENS,
                "thinking": thinking_config,
                "messages": [{"role": "user", "content": prompt}]
            }
            
            # Add effort parameter for Opus 4.6 adaptive thinking
            if USE_ADAPTIVE_THINKING:
                request_kwargs["output_config"] = {"effort": EFFORT_LEVEL}
            
            with client.messages.stream(**request_kwargs) as stream:
                for event in stream:
                    pass  # Just consume the stream
                
                # Get final message after stream completes
                response = stream.get_final_message()
            
            # Check for refusal FIRST
            if response.stop_reason == "refusal":
                print(f"REFUSED (row {row_idx}): Model refused to answer (safety filter)")
                refused_questions.append({"row": row_idx, "reason": "refusal"})
                return "[REFUSED - Safety filter]", 0, 0, 0, "[Model refused to answer]"
            
            # Extract text AND thinking from response.content
            text = ""
            thinking_content = ""
            
            for block in response.content:
                block_type = getattr(block, 'type', None)
                
                if block_type == "thinking":
                    # Try different attribute names for thinking content
                    thinking_content = (
                        getattr(block, 'thinking', None) or 
                        getattr(block, 'text', None) or 
                        getattr(block, 'content', None) or 
                        ''
                    )
                elif block_type == "text":
                    text = (getattr(block, 'text', '') or '').strip()
                elif block_type == "tool_use":
                    # Skip tool use blocks
                    pass
                else:
                    # Unknown block type - log it
                    print(f"Unknown block type: {block_type}")
            
            # Extract token usage from response.usage
            input_tokens = getattr(response.usage, 'input_tokens', 0) or 0
            output_tokens = getattr(response.usage, 'output_tokens', 0) or 0
            
            # Estimate thinking tokens from summary text
            # Note: This is from the SUMMARY, not the full thinking 
            thinking_tokens_est = int(len(thinking_content.split()) * 1.3) if thinking_content else 0
            
            if text:
                return text, input_tokens, output_tokens, thinking_tokens_est, thinking_content
            
            # Empty response - increment retry counter
            retry_count += 1
            print(f"Empty response (row {row_idx}) — retry {retry_count}/{MAX_RETRIES}, stop_reason={response.stop_reason}")
            time.sleep(4)

        except anthropic.RateLimitError as e:
            print(f"Rate limit — retrying in 10s: {e}")
            time.sleep(10)
            # Don't increment retry_count for rate limits
        except anthropic.APIError as e:
            retry_count += 1
            print(f"API issue ({retry_count}/{MAX_RETRIES}) — retrying in 4s: {e}")
            time.sleep(4)
        except Exception as e:
            retry_count += 1
            print(f"Error ({retry_count}/{MAX_RETRIES}) — retrying in 4s: {e}")
            time.sleep(4)
    
    # Max retries exceeded
    print(f"FAILED (row {row_idx}): Max retries ({MAX_RETRIES}) exceeded")
    refused_questions.append({"row": row_idx, "reason": "max_retries"})
    return "[FAILED - Max retries]", 0, 0, 0, ""

# ============ WORKER ============
row_lock = threading.Lock()
progress = {"count": 0}

def process_row(i):
    if pd.notna(df.at[i, 'answer_to_prompt_1']) and pd.notna(df.at[i, 'answer_to_prompt_2']):
        return

    p1 = str(df.at[i, 'prompt.1'])
    p2 = str(df.at[i, 'prompt.2'])
    opt = str(df.at[i, 'OPTIONS'])

    if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
    if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

    a1, in1, out1, think1, think_content1 = ask_until_success(p1, row_idx=i)
    a2, in2, out2, think2, think_content2 = ask_until_success(p2, row_idx=i)

    with row_lock:
        df.at[i, 'answer_to_prompt_1'] = a1
        df.at[i, 'answer_to_prompt_2'] = a2
        df.at[i, 'input_tokens_1'] = in1
        df.at[i, 'input_tokens_2'] = in2
        df.at[i, 'output_tokens_1'] = out1
        df.at[i, 'output_tokens_2'] = out2
        df.at[i, 'thinking_tokens_1'] = think1
        df.at[i, 'thinking_tokens_2'] = think2
        df.at[i, 'thinking_content_1'] = think_content1
        df.at[i, 'thinking_content_2'] = think_content2
        progress["count"] += 1

        if progress["count"] % SAVE_EVERY == 0:
            save()
            print(f"Saved @ {progress['count']} rows | Thinking: ~{think1}, ~{think2} tokens")

# ============ RUN ============
todo = [i for i in range(len(df))
        if pd.isna(df.at[i, 'answer_to_prompt_1']) or pd.isna(df.at[i, 'answer_to_prompt_2'])]

print(f"{len(todo)} rows to process")
print(f"Model: {model_name}")
if USE_ADAPTIVE_THINKING:
    print(f"Thinking: Adaptive (controlled by effort level)")
else:
    print(f"Thinking budget: {THINKING_BUDGET} tokens")
print(f"Effort level: {EFFORT_LEVEL}")
print(f"Max tokens: {MAX_TOKENS}")
print(f"Max retries: {MAX_RETRIES}")
print(f"Output: {CSV_PATH}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, i) for i in todo]
    for idx, _ in enumerate(as_completed(futures), 1):
        if idx % 10 == 0:
            print(f"Progress: {idx}/{len(todo)}")

save()

# Report refused/failed questions
if refused_questions:
    print(f"\n{len(refused_questions)} questions REFUSED or FAILED:")
    for rq in refused_questions:
        print(f"   Row {rq['row']}: {rq['reason']}")

print(f"\nCOMPLETE — Saved to {CSV_PATH}")